# HP (HEALPix) model output inspection

Loads raw predictions and targets from trained HP (HEALPix) climate models.
Companion to `check_outputs_from_models_nohp.ipynb` (lat/lon grid models).

## Cross-notebook comparison
Set `TARGET_YEAR` and `TARGET_MONTH` to the same values in both notebooks to inspect
the identical timestep from both model families side by side.

In [1]:
import sys
sys.path.insert(0, "/home/x_tagty/equivariant-posteriors")

import numpy as np
import matplotlib.pyplot as plt
import healpy as hp

from experiments.climate.notebooks.sample_predictions import (
    get_sample_predictions,
    get_predictions_for_timestamp,
)
from experiments.climate.evaluation.evaluate_climate_hp import load_create_config
from experiments.climate.evaluation.timestamp_utils import (
    sample_id_to_timestamp,
    format_timestamp,
    MONTH_NAMES,
)

/home/x_tagty/equivariant-posteriors/.venv/lib64/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/x_tagty/equivariant-posteriors/.venv/lib64/python3.11/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


[Compute environment] paths: 
[Paths] checkpoints: /proj/heal_pangu/eqp_climate/checkpoints (/proj/heal_pangu/eqp_climate/checkpoints)
[Paths] locks: locks (/home/x_tagty/equivariant-posteriors/experiments/climate/notebooks/locks)
[Paths] distributed_requests: distributed_requests (/home/x_tagty/equivariant-posteriors/experiments/climate/notebooks/distributed_requests)
[Paths] artifacts: /proj/heal_pangu/eqp_climate/artifacts (/proj/heal_pangu/eqp_climate/artifacts)
[Paths] datasets: /proj/heal_pangu/users/x_tagty/climateset (/proj/heal_pangu/users/x_tagty/climateset)
[Compute environment] envs: None


In [ ]:

import glob, os, io, contextlib, duckdb

from lib.paths import get_checkpoint_path
from experiments.climate.data.climateset_data_hp import ClimatesetDataHP


def _query_all_steps(checkpoint_dir, metric_name):
    """Return list of (step, mean_float) from all duck_*.db files in checkpoint_dir."""
    db_files = sorted(glob.glob(os.path.join(str(checkpoint_dir), "duck_*.db")))
    rows = []
    for db_path in db_files:
        try:
            con = duckdb.connect(db_path, read_only=True)
            rows.extend(con.execute(
                "SELECT step, mean_float FROM checkpoint_sample_metric "
                "WHERE name = ? AND mean_float IS NOT NULL",
                [metric_name],
            ).fetchall())
            con.close()
        except Exception:
            pass
    return rows


def get_best_epoch_hp(curried, seed=0, metric_name="rmse_overall"):
    """Return (best_epoch, best_val) considering only epochs whose checkpoint files exist."""
    train_run = curried(ensemble_id=seed)
    checkpoint_dir = get_checkpoint_path(train_run.train_config)
    rows = _query_all_steps(checkpoint_dir, metric_name)
    if not rows:
        return None, None
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        train_ds = ClimatesetDataHP(train_run.train_config.train_data_config)
    n_train = len(train_ds)
    best_epoch, best_val = None, float("inf")
    for step, val in rows:
        epoch = round(step / n_train)
        if (checkpoint_dir / f"model_epoch_{epoch:04d}").is_file() and val < best_val:
            best_val, best_epoch = val, epoch
    return best_epoch, best_val


def sample_rmse(r, var_name):
    """Simple spatial RMSE for a single-timestep result dict (prediction/target shape (C, P))."""
    vi   = r["var_names"].index(var_name)
    pred = np.asarray(r["prediction"][vi]).ravel()
    tgt  = np.asarray(r["target"][vi]).ravel()
    return float(np.sqrt(np.mean((pred - tgt) ** 2)))


## Config — edit these

Set `TARGET_YEAR` / `TARGET_MONTH` to the same values used in the nohp notebook
(`check_outputs_from_models_nohp.ipynb`) to compare the exact same timestep.

In [ ]:
#CONFIG = "/home/x_tagty/equivariant-posteriors/experiments/climate/persisted_configs/final_climate_pear_runs/train_climate_pear_multiseed.py"
CONFIG = "/home/x_tagty/equivariant-posteriors/experiments/climate/persisted_configs/var_weighing/onevar_trunc_init_model_ablation.py"


# Climate model indices to compare (0-14)
# 0=AWI-CM-1-1-MR  2=CAS-ESM2-0  7=GFDL-ESM4  12=NorESM2-LM
CLIMATE_MODEL_IDXS = [2, 7, 12]

EPOCH = None  # None → auto-select best epoch from DuckDB; set an int to pin manually
SEED  = 3
LR    = 2e-4

# ---- Cross-notebook timestamp anchor ----
# Set these to the same values in check_outputs_from_models_nohp.ipynb
# to guarantee you are looking at the identical timestep.
TARGET_YEAR  = 2040
TARGET_MONTH = 4    # 1=Jan ... 12=Dec

# Output variable for map panels
VAR_NAME = "pr"    # or "tas"

# For the multi-batch exploratory cells below
N_BATCHES = 2   # 2 x 12 = 24 samples

## 1 · Exact timestep comparison — same sample as nohp notebook

In [ ]:
base_create_config = load_create_config(CONFIG)

ts_results = {}
for idx in CLIMATE_MODEL_IDXS:
    def make_curried(climate_idx):
        return lambda ensemble_id, **kw: base_create_config(
            ensemble_id=ensemble_id,
            climate_model_idx=climate_idx,
            lr=LR,
            **kw,
        )
    curried = make_curried(idx)
    _ep = EPOCH
    if _ep is None:
        _ep, _best_val = get_best_epoch_hp(curried, seed=SEED)
        if _ep is None:
            raise RuntimeError(
                f"No DuckDB metrics found for climate_model_idx={idx}. Set EPOCH manually."
            )
        print(f"  [idx={idx}] best epoch from DuckDB: {_ep}  (val RMSE={_best_val:.4e})")
    print(f"Loading idx={idx}, epoch={_ep}, target={MONTH_NAMES[TARGET_MONTH-1]} {TARGET_YEAR} ...")
    r = get_predictions_for_timestamp(
        curried, _ep, year=TARGET_YEAR, month=TARGET_MONTH, variant_idx=SEED
    )
    r["used_epoch"] = _ep
    r["sample_rmse"] = sample_rmse(r, VAR_NAME)
    ts_results[idx] = r
    print(f"  -> {r['model_name']}  timestamp: {r['timestamp_label']}  "
          f"prediction shape: {r['prediction'].shape}  RMSE({VAR_NAME})={r['sample_rmse']:.4e}")

print("\nAll loaded. Variables:", ts_results[CLIMATE_MODEL_IDXS[0]]["var_names"])

### Single model — True / Predicted / Error

In [ ]:
def plot_true_pred_error(r, var_name, fig_title=None, mollview=False):
    var_idx  = r["var_names"].index(var_name)
    true_map = r["target"][var_idx]       # (P,)
    pred_map = r["prediction"][var_idx]   # (P,)
    err_map  = r["error"][var_idx]        # (P,)

    unit = "K" if var_name == "tas" else "kg/m2/s"
    vmin, vmax = true_map.min(), true_map.max()

    fig = plt.figure(figsize=(18, 5))
    if fig_title:
        fig.suptitle(fig_title, fontsize=13)

    if mollview:
        hp.mollview(true_map, nest=True, fig=fig, sub=(1, 3, 1),
                    title=f"True {var_name} [{unit}]", min=vmin, max=vmax, cmap="RdBu_r")
        hp.mollview(pred_map, nest=True, fig=fig, sub=(1, 3, 2),
                    title=f"Predicted {var_name} [{unit}]", min=vmin, max=vmax, cmap="RdBu_r")
        hp.mollview(err_map,  nest=True, fig=fig, sub=(1, 3, 3),
                    title=f"Abs Error {var_name} [{unit}]", cmap="YlOrRd")
    else:
        hp.cartview(true_map, nest=True, fig=fig, sub=(1, 3, 1),
                    title=f"True {var_name} [{unit}]", min=vmin, max=vmax, cmap="RdBu_r", flip="geo")
        hp.cartview(pred_map, nest=True, fig=fig, sub=(1, 3, 2),
                    title=f"Predicted {var_name} [{unit}]", min=vmin, max=vmax, cmap="RdBu_r", flip="geo")
        hp.cartview(err_map,  nest=True, fig=fig, sub=(1, 3, 3),
                    title=f"Abs Error {var_name} [{unit}]", cmap="YlOrRd", flip="geo")
    plt.tight_layout()
    plt.show()


idx0 = CLIMATE_MODEL_IDXS[0]
r0   = ts_results[idx0]
_ep0 = r0.get("used_epoch", EPOCH)
_rm0 = r0.get("sample_rmse")
_rm_s = f"  RMSE={_rm0:.3e}" if _rm0 is not None else ""
plot_true_pred_error(
    r0, VAR_NAME,
    fig_title=f"{r0['model_name']}  {r0['timestamp_label']}  seed={SEED}  ep={_ep0}{_rm_s}"
)

### Multi-model comparison at the same timestep

In [ ]:
n_models = len(CLIMATE_MODEL_IDXS)
fig = plt.figure(figsize=(18, 5 * n_models))
fig.suptitle(
    f"{VAR_NAME}  —  {format_timestamp('ssp245', TARGET_YEAR, TARGET_MONTH)}  seed={SEED}",
    fontsize=14
)

for row, idx in enumerate(CLIMATE_MODEL_IDXS):
    r = ts_results[idx]
    var_idx  = r["var_names"].index(VAR_NAME)
    true_map = r["target"][var_idx]
    pred_map = r["prediction"][var_idx]
    err_map  = r["error"][var_idx]
    vmin, vmax = true_map.min(), true_map.max()
    name = r["model_name"]
    _ep  = r.get("used_epoch", EPOCH)
    _rm  = r.get("sample_rmse")
    _rm_s = f"  RMSE={_rm:.3e}" if _rm is not None else ""

    hp.mollview(true_map, nest=True, fig=fig, sub=(n_models, 3, row * 3 + 1),
                title=f"{name}  ep={_ep}{_rm_s}\nTrue", min=vmin, max=vmax, cmap="RdBu_r")
    hp.mollview(pred_map, nest=True, fig=fig, sub=(n_models, 3, row * 3 + 2),
                title=f"{name}\nPredicted", min=vmin, max=vmax, cmap="RdBu_r")
    hp.mollview(err_map,  nest=True, fig=fig, sub=(n_models, 3, row * 3 + 3),
                title=f"{name}\nAbs Error", cmap="YlOrRd")

plt.tight_layout()
plt.show()

---
## 2 · Exploratory multi-batch section

Load `N_BATCHES` worth of samples and inspect error distributions / spatial maps.

In [ ]:
results = {}
for idx in CLIMATE_MODEL_IDXS:
    def make_curried(climate_idx):
        return lambda ensemble_id, **kw: base_create_config(
            ensemble_id=ensemble_id,
            climate_model_idx=climate_idx,
            lr=LR,
            **kw,
        )
    curried = make_curried(idx)
    print(f"Loading idx={idx}, {N_BATCHES} batches ...")
    results[idx] = get_sample_predictions(curried, EPOCH, variant_idx=SEED, n_batches=N_BATCHES)
    r = results[idx]
    # Print what timesteps were loaded (first and last sample_id)
    first_sid = r["sample_ids"][0]
    last_sid  = r["sample_ids"][-1]
    sc0, y0, m0 = sample_id_to_timestamp(first_sid, r["test_data_config"])
    sc1, y1, m1 = sample_id_to_timestamp(last_sid,  r["test_data_config"])
    print(f"  -> {r['model_name']}  predictions shape: {r['predictions'].shape}")
    print(f"     covers {format_timestamp(sc0, y0, m0)} -> {format_timestamp(sc1, y1, m1)}")

print("Done.")

### Error distributions

In [ ]:
var_names = results[CLIMATE_MODEL_IDXS[0]]["var_names"]
fig, axes = plt.subplots(1, len(var_names), figsize=(7 * len(var_names), 4))
if not hasattr(axes, "__len__"):
    axes = [axes]

for var_idx, var_name in enumerate(var_names):
    ax = axes[var_idx]
    for idx in CLIMATE_MODEL_IDXS:
        r = results[idx]
        err_flat = r["errors"][:, var_idx, :].ravel()
        ax.hist(err_flat, bins=80, alpha=0.5, label=r["model_name"], density=True)
    ax.set_xlabel(f"Abs error  {var_name}")
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)
    ax.set_title(f"{var_name} error distribution")

plt.tight_layout()
plt.show()

### Mean spatial error map (averaged over all loaded samples)

In [ ]:
n_models  = len(CLIMATE_MODEL_IDXS)
var_names = results[CLIMATE_MODEL_IDXS[0]]["var_names"]
n_vars    = len(var_names)

fig = plt.figure(figsize=(8 * n_vars, 4 * n_models))
fig.suptitle("Mean spatial abs error  (HP)", fontsize=13)

for row, idx in enumerate(CLIMATE_MODEL_IDXS):
    r = results[idx]
    for col, var_name in enumerate(var_names):
        mean_err = r["errors"][:, col, :].mean(axis=0)  # (P,)
        hp.mollview(mean_err, nest=True, fig=fig,
                    sub=(n_models, n_vars, row * n_vars + col + 1),
                    title=f"{r['model_name']}  {var_name}",
                    cmap="YlOrRd")

plt.tight_layout()
plt.show()